In [ ]:
import pandas as pd
import numpy as np 
# NEED TO CLEAN THIS UP 

# POOR MAN'S RELOOP
Regress outcome on covariates, pair score, and treatment indicator 

In [1]:
def get_pair_score(journal, outcome = "Times cited (36mo)"):
    filename = f"open_access_results/{journal}_qualities.csv"
    results = pd.read_csv(filename)
    results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

        transform_df = pd.concat([
            results.loc[results[col] == 'Paper 1', 'Article Title.1'],
            results.loc[results[col] == 'Paper 2', 'Article Title.2']
        ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', f'{col}_score']

        journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
    filename = f"open_access_results/{journal}_basic.csv"
    results = pd.read_csv(filename)
    transform_df = pd.concat([
        results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
        results.loc[results['response'] == 'Paper 2', 'Article Title.2']
        ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', 'risk_score']
    rate_df.sort_values('Article Title').head()
    rate_df = rate_df[['Article Title', 'risk_score']]

    journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

    # get other covariates
    journal_df['log_authors'] = np.log(journal_df['Authors'])
    journal_df['log_pages'] = np.log(journal_df['Page length'])
    journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
    journal_df['log_outcome'] = np.log(journal_df[outcome] + 1)
    return journal_df

In [15]:
import statsmodels.api as sm
def get_ses(journal): 
    if journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
        journal_df = get_pair_score(journal)
    y = journal_df['log_outcome']
    X1 = sm.add_constant(journal_df[['Open Access', 'log_authors', 'log_pages', 'review']])
    X2 = sm.add_constant(journal_df[['Open Access', 'risk_score', 'log_authors', 'log_pages', 'review']])
    X3 = sm.add_constant(journal_df[['Open Access', 'topic_novelty_score', 'topic_popularity_score', 'title_catchiness_score', 'generalizability_score', 
                                      'writing_quality_score', 'impact_of_results_score', 'subfield_popularity_score', 'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'log_authors', 'log_pages', 'review']])
    X4 = sm.add_constant(journal_df[['Open Access', 'risk_score', 'topic_novelty_score', 'topic_popularity_score', 'title_catchiness_score', 'generalizability_score', 
                                      'writing_quality_score', 'impact_of_results_score', 'subfield_popularity_score', 'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'log_authors', 'log_pages', 'review']])

    model1 = sm.OLS(y, X1).fit()
    model2 = sm.OLS(y, X2).fit()
    model3 = sm.OLS(y, X3).fit()
    model4 = sm.OLS(y, X4).fit()

    return [model1.bse['Open Access'], model2.bse['Open Access'], model3.bse['Open Access'], model4.bse['Open Access']]

rows = []
for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
    #print(journal)
    row = get_ses(journal)
    rows.append(row)

se_results = pd.DataFrame(rows, index = ['Science', 'Neurophysiology', 'Genetics', 'FASEB','Applied Physiology'], columns =['Base Covariates', 'Base + Rating Score', 'Base + 10 Qualities', 'Base + Both'] )
print(se_results)

                    Base Covariates  Base + Rating Score  Base + 10 Qualities  \
Science                    0.115634             0.104369             0.103648   
Neurophysiology            0.110551             0.103870             0.105060   
Genetics                   0.101104             0.097396             0.095202   
FASEB                      0.094287             0.088074             0.092397   
Applied Physiology         0.156371             0.139208             0.144166   

                    Base + Both  
Science                0.099566  
Neurophysiology        0.104709  
Genetics               0.095206  
FASEB                  0.087231  
Applied Physiology     0.138434  


In [16]:
se_results['Base Covariates']/se_results['Base + Both'] - 1

Science               0.161381
Neurophysiology       0.055796
Genetics              0.061942
FASEB                 0.080888
Applied Physiology    0.129567
dtype: float64

In [24]:
# get all journals together

full_df = pd.DataFrame()

for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
    journal_df = get_pair_score(journal)
    journal_df['journal'] = journal
    full_df = pd.concat([full_df, journal_df])
full_df = pd.concat([full_df, pd.get_dummies(full_df["journal"], drop_first=True).astype(float)], axis=1)
#print(full_df.columns)
#print(full_df.dtypes)
y = full_df['log_outcome']
X1 = sm.add_constant(full_df[['Open Access', 'genetics', 'neuro', 'physio', 'science', 'log_authors', 'log_pages', 'review']])
X2 = sm.add_constant(full_df[['Open Access', 'genetics', 'neuro', 'physio', 'science','risk_score', 'log_authors', 'log_pages', 'review']])
X3 = sm.add_constant(full_df[['Open Access', 'genetics', 'neuro', 'physio', 'science','topic_novelty_score', 'topic_popularity_score', 'title_catchiness_score', 'generalizability_score', 
                                      'writing_quality_score', 'impact_of_results_score', 'subfield_popularity_score', 'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'log_authors', 'log_pages', 'review']])
X4 = sm.add_constant(full_df[['Open Access', 'genetics', 'neuro', 'physio', 'science','risk_score', 'topic_novelty_score', 'topic_popularity_score', 'title_catchiness_score', 'generalizability_score', 
                                      'writing_quality_score', 'impact_of_results_score', 'subfield_popularity_score', 'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'log_authors', 'log_pages', 'review']])

model1 = sm.OLS(y, X1).fit()
model2 = sm.OLS(y, X2).fit()
model3 = sm.OLS(y, X3).fit()
model4 = sm.OLS(y, X4).fit()

print([model1.bse['Open Access'], model2.bse['Open Access'], model3.bse['Open Access'], model4.bse['Open Access']])
    

[np.float64(0.051304404872514565), np.float64(0.04738461472991526), np.float64(0.047794757987249116), np.float64(0.04655896595192256)]


# FULL RELOOP

In [33]:
# test on just science
import statsmodels.api as sm 
journal = "science"
outcome = "Times cited (36mo)"

filename = f"open_access_results/{journal}_qualities.csv"
results = pd.read_csv(filename)
results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

    transform_df = pd.concat([
        results.loc[results[col] == 'Paper 1', 'Article Title.1'],
        results.loc[results[col] == 'Paper 2', 'Article Title.2']
    ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', f'{col}_score']

    journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
filename = f"open_access_results/{journal}_basic.csv"
results = pd.read_csv(filename)
transform_df = pd.concat([
    results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
    results.loc[results['response'] == 'Paper 2', 'Article Title.2']
    ])

rate_df = transform_df.value_counts().reset_index()
rate_df.columns = ['Article Title', 'risk_score']
rate_df.sort_values('Article Title').head()
rate_df = rate_df[['Article Title', 'risk_score']]

journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

    # first model - neither 
journal_df['log_authors'] = np.log(journal_df['Authors'])
journal_df['log_pages'] = np.log(journal_df['Page length'])
journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
journal_df['log_outcome'] = np.log(journal_df[outcome] + 1)


X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'Open Access']].reset_index(drop = True)
y = journal_df['log_outcome']

journal_df['y_it'] = np.nan
journal_df['y_ic'] = np.nan

# for each observation, drop observation 
# refit model 
# predict for this observation with open access = 1 and = 0
for i in journal_df.index:
    X_withouti = sm.add_constant(X.drop(index = i))
    y_withouti = np.delete(y, i)
    model_withouti = sm.OLS(y_withouti, X_withouti).fit()

    observation_it = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'Open Access']].iloc[[i]]
    observation_it.loc[:,'Open Access'] = 1 

    observation_ic = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'Open Access']].iloc[[i]]
    observation_ic.loc[:,'Open Access'] = 0

    i_t = sm.add_constant(observation_it, has_constant = 'add')
    i_c = sm.add_constant(observation_ic, has_constant = 'add')

    y_it = model_withouti.predict(i_t).iloc[0]
    y_ic = model_withouti.predict(i_c).iloc[0]
    
    journal_df.loc[i, 'y_ic'] = y_ic
    journal_df.loc[i, 'y_it'] = y_it


    

In [34]:
journal_df.head()

,Journal,Year,Volume,Issue,Beginning page,Article Title,Authors,Article type,No. References,Page length,...,meaningful_contributions_score,journal_fit_score,applicability_score,risk_score,log_authors,log_pages,review,log_outcome,y_it,y_ic
0,Science,2007,316,5829,1298,Large-scale spatial-transmission models of inf...,1,Review,35,4,...,351,220,141,373,0.000000,1.386294,1,3.970292,3.694863,3.551687
1,Science,2007,316,5829,1302,Seawater chemistry and early carbonate biomine...,1,Article,11,1,...,356,198,71,110,0.000000,0.000000,0,2.995732,2.427246,2.283557
2,Science,2007,316,5829,1303,"155,000 years of West African monsoon and ocea...",4,Article,47,5,...,363,231,94,127,1.386294,1.609438,0,3.637586,4.038470,3.898461
3,Science,2007,316,5829,1307,Legumes symbioses: Absence of Nod genes in pho...,34,Article,41,6,...,337,138,96,130,3.526361,1.791759,0,4.369448,4.895350,4.755785
4,Science,2007,316,5829,1312,Quantum register based on individual electroni...,9,Article,28,5,...,311,298,112,281,2.197225,1.609438,0,4.727388,4.311506,4.169362


In [ ]:
# need p for each observation

,log_authors,log_pages,review,Self-archived,Open Access
0,0.0,1.386294,1,0,1


In [17]:
model_withouti.model.exog_names

['const', 'log_authors', 'log_pages', 'review', 'Self-archived', 'Open Access']

In [13]:
test = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'Open Access']].iloc[0]
test.loc['Open Access'] = 0
test

log_authors      0.000000
log_pages        1.386294
review           1.000000
Self-archived    0.000000
Open Access      0.000000
Name: 0, dtype: float64

In [ ]:
# get y_it and y_ic imputations 
# for the models I already have in step 3, add in treatment and control (do i need to make these linear models instead for that to work?)
# yes, switch this to a linear model 


def get_imputations(journal, outcome = "Times cited (36mo)"):
    filename = f"open_access_results/{journal}_qualities.csv"
    results = pd.read_csv(filename)
    results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

        transform_df = pd.concat([
            results.loc[results[col] == 'Paper 1', 'Article Title.1'],
            results.loc[results[col] == 'Paper 2', 'Article Title.2']
        ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', f'{col}_score']

        journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
    filename = f"open_access_results/{journal}_basic.csv"
    results = pd.read_csv(filename)
    transform_df = pd.concat([
        results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
        results.loc[results['response'] == 'Paper 2', 'Article Title.2']
        ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', 'risk_score']
    rate_df.sort_values('Article Title').head()
    rate_df = rate_df[['Article Title', 'risk_score']]

    journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

    # first model - neither 
    journal_df['log_authors'] = np.log(journal_df['Authors'])
    journal_df['log_pages'] = np.log(journal_df['Page length'])
    journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
    journal_df['log_outcome'] = np.log(journal_df[outcome] + 1)


    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'Open Access']]
    y = journal_df['log_outcome']

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds_neither'] = rf.oob_prediction_ 

    # second - risk score only 
    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'risk_score']]

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds_basic'] = rf.oob_prediction_ 

    # third - qualities only 
    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score']]

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds_qualities'] = rf.oob_prediction_ 

    #fourth - both

    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'risk_score']]

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds_both'] = rf.oob_prediction_ 
   

    return journal_df 

# OLD STUFF

In [2]:
def get_basic_correlation_results(journal, outcome = "Times cited (36mo)"):
    filename = f"open_access_results/{journal}_basic.csv"
    results = pd.read_csv(filename)
    transform_df = pd.concat([
    results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
    results.loc[results['response'] == 'Paper 2', 'Article Title.2']
    ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', 'risk_score']
    rate_df.sort_values('Article Title').head()

    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    results_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left')
    results_df['log_outcome'] = np.log(results_df[outcome] + 1)
    basic_corr = results_df['log_outcome'].corr(results_df['risk_score'])
    return(basic_corr)

def get_qualities_correlation_results(journal, outcome = "Times cited (36mo)"):
    filename = f"open_access_results/{journal}_qualities.csv"
    results = pd.read_csv(filename)
    results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

        transform_df = pd.concat([
            results.loc[results[col] == 'Paper 1', 'Article Title.1'],
            results.loc[results[col] == 'Paper 2', 'Article Title.2']
        ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', f'{col}_score']

        journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left')
    journal_df['log_outcome'] = np.log(journal_df[outcome] + 1)
    quality_correlations = journal_df[['topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score',
       'journal_fit_score', 'applicability_score']].corrwith(journal_df['log_outcome'])
    return(quality_correlations)

from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression


def run_random_forests(journal, outcome = "Times cited (36mo)"):
    filename = f"open_access_results/{journal}_qualities.csv"
    results = pd.read_csv(filename)
    results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

        transform_df = pd.concat([
            results.loc[results[col] == 'Paper 1', 'Article Title.1'],
            results.loc[results[col] == 'Paper 2', 'Article Title.2']
        ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', f'{col}_score']

        journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
    filename = f"open_access_results/{journal}_basic.csv"
    results = pd.read_csv(filename)
    transform_df = pd.concat([
        results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
        results.loc[results['response'] == 'Paper 2', 'Article Title.2']
        ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', 'risk_score']
    rate_df.sort_values('Article Title').head()
    rate_df = rate_df[['Article Title', 'risk_score']]

    journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

    # first random forest - neither 
    journal_df['log_authors'] = np.log(journal_df['Authors'])
    journal_df['log_pages'] = np.log(journal_df['Page length'])
    journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
    journal_df['log_outcome'] = np.log(journal_df[outcome] + 1)


    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived']]
    y = journal_df['log_outcome']

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds_neither'] = rf.oob_prediction_ 

    # second - risk score only 
    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'risk_score']]

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds_basic'] = rf.oob_prediction_ 

    # third - qualities only 
    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score']]

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds_qualities'] = rf.oob_prediction_ 

    #fourth - both

    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'risk_score']]

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   # must be True for OOB
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds_both'] = rf.oob_prediction_ 
   

    return journal_df 

    


In [3]:
rows = []
for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
    journal_df = run_random_forests(journal)
    neither_corr = journal_df['log_outcome'].corr(journal_df['oob_preds_neither'])
    basic_corr = journal_df['log_outcome'].corr(journal_df['oob_preds_basic'])
    qualities_corr = journal_df['log_outcome'].corr(journal_df['oob_preds_qualities'])
    both_corr = journal_df['log_outcome'].corr(journal_df['oob_preds_both'])
    row = [neither_corr, basic_corr, qualities_corr, both_corr]
    rows.append(row)

rf_results = pd.DataFrame(rows, index = ['Science', 'Neurophysiology', 'Genetics', 'FASEB','Applied Physiology'], columns =['Base Covariates', 'Base + Rating Score', 'Base + 11 Qualities', 'Base + Both'] )
print(rf_results)

                    Base Covariates  Base + Rating Score  Base + 11 Qualities  \
Science                    0.402611             0.514239             0.574734   
Neurophysiology            0.135365             0.330803             0.303937   
Genetics                   0.190359             0.206164             0.284343   
FASEB                      0.192006             0.260009             0.208367   
Applied Physiology         0.193243             0.433162             0.384400   

                    Base + Both  
Science                0.617976  
Neurophysiology        0.345577  
Genetics               0.291995  
FASEB                  0.353867  
Applied Physiology     0.459473  


In [23]:
def run_rf_full(outcome = "Times cited (36mo)"):
    all_journals = pd.DataFrame()

    for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
        filename = f"open_access_results/{journal}_qualities.csv"
        results = pd.read_csv(filename)
        results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
        journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
        for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

            transform_df = pd.concat([
                results.loc[results[col] == 'Paper 1', 'Article Title.1'],
                results.loc[results[col] == 'Paper 2', 'Article Title.2']
            ])

            rate_df = transform_df.value_counts().reset_index()
            rate_df.columns = ['Article Title', f'{col}_score']

            journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
        filename = f"open_access_results/{journal}_basic.csv"
        results = pd.read_csv(filename)
        transform_df = pd.concat([
            results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
            results.loc[results['response'] == 'Paper 2', 'Article Title.2']
            ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', 'risk_score']
        rate_df.sort_values('Article Title').head()
        rate_df = rate_df[['Article Title', 'risk_score']]

        journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

        journal_df['log_authors'] = np.log(journal_df['Authors'])
        journal_df['log_pages'] = np.log(journal_df['Page length'])
        journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
        journal_df['log_outcome'] = np.log(journal_df["Times cited (36mo)"] + 1)
        journal_df['journal'] = journal

        all_journals = pd.concat([all_journals, journal_df])

        all_journals['science'] = np.where(all_journals['journal'] == "science", 1, 0)
        all_journals['neuro'] = np.where(all_journals['journal'] == "neuro", 1, 0)
        all_journals['genetics'] = np.where(all_journals['journal'] == "genetics", 1, 0)
        all_journals['faseb'] = np.where(all_journals['journal'] == "faseb", 1, 0)

        # first random forest - neither 
        all_journals['log_authors'] = np.log(all_journals['Authors'])
        all_journals['log_pages'] = np.log(all_journals['Page length'])
        all_journals['review'] = np.where(all_journals['Article type'] == "Review", 1, 0) 
        all_journals['log_outcome'] = np.log(all_journals[outcome] + 1)


        X = all_journals[['log_authors', 'log_pages', 'review', 'Self-archived', 'science', 'neuro', 'genetics', 'faseb']]
        y = all_journals['log_outcome']

        rf = RandomForestRegressor(
            n_estimators=500,
            oob_score=True,
            bootstrap=True,   # must be True for OOB
            random_state=42
        )

        rf.fit(X, y)

        all_journals['oob_preds_neither'] = rf.oob_prediction_ 

        # second - risk score only 
        X = all_journals[['log_authors', 'log_pages', 'review', 'Self-archived', 'science', 'neuro', 'genetics', 'faseb', 'risk_score']]

        rf = RandomForestRegressor(
            n_estimators=500,
            oob_score=True,
            bootstrap=True,   # must be True for OOB
            random_state=42
        )

        rf.fit(X, y)

        all_journals['oob_preds_basic'] = rf.oob_prediction_ 

        # third - qualities only 
        X = all_journals[['log_authors', 'log_pages', 'review', 'Self-archived', 'science', 'neuro', 'genetics', 'faseb', 'topic_novelty_score',
        'topic_popularity_score', 'title_catchiness_score',
        'generalizability_score', 'writing_quality_score',
        'impact_of_results_score', 'subfield_popularity_score',
        'technicality_score', 'meaningful_contributions_score', 'applicability_score']]

        rf = RandomForestRegressor(
            n_estimators=500,
            oob_score=True,
            bootstrap=True,   # must be True for OOB
            random_state=42
        )

        rf.fit(X, y)

        all_journals['oob_preds_qualities'] = rf.oob_prediction_ 

        #fourth - both

        X = all_journals[['log_authors', 'log_pages', 'review', 'Self-archived', 'science', 'neuro', 'genetics', 'faseb', 'topic_novelty_score',
        'topic_popularity_score', 'title_catchiness_score',
        'generalizability_score', 'writing_quality_score',
        'impact_of_results_score', 'subfield_popularity_score',
        'technicality_score', 'meaningful_contributions_score', 'applicability_score', 'risk_score']]

        rf = RandomForestRegressor(
            n_estimators=500,
            oob_score=True,
            bootstrap=True,   # must be True for OOB
            random_state=42
        )

        rf.fit(X, y)

        all_journals['oob_preds_both'] = rf.oob_prediction_ 

        return all_journals

        neither_corr = all_journals['log_outcome'].corr(all_journals['oob_preds_neither'])
        basic_corr = all_journals['log_outcome'].corr(all_journals['oob_preds_basic'])
        qualities_corr = all_journals['log_outcome'].corr(all_journals['oob_preds_qualities'])
        both_corr = all_journals['log_outcome'].corr(all_journals['oob_preds_both'])
        row = [neither_corr, basic_corr, qualities_corr, both_corr]

        return row

In [24]:
all_journals = run_rf_full()
neither_corr = all_journals['log_outcome'].corr(all_journals['oob_preds_neither'])
basic_corr = all_journals['log_outcome'].corr(all_journals['oob_preds_basic'])
qualities_corr = all_journals['log_outcome'].corr(all_journals['oob_preds_qualities'])
both_corr = all_journals['log_outcome'].corr(all_journals['oob_preds_both'])
full_results = [neither_corr, basic_corr, qualities_corr, both_corr]

print(full_results)

[np.float64(0.4034415108859438), np.float64(0.5116493014135045), np.float64(0.5751577846418833), np.float64(0.6181780196964521)]


In [25]:
new_index = "All Journals"
rf_results.loc[new_index] =  full_results
rf_results

,Base Covariates,Base + Rating Score,Base + 11 Qualities,Base + Both
Science,0.402611,0.514239,0.574734,0.617976
Neurophysiology,0.135365,0.330803,0.303937,0.345577
Genetics,0.190359,0.206164,0.284343,0.291995
FASEB,0.192006,0.260009,0.208367,0.353867
Applied Physiology,0.193243,0.433162,0.384400,0.459473
All Journals,0.403442,0.511649,0.575158,0.618178


In [22]:
print(rf_results.to_latex())

\begin{tabular}{lrrrr}
\toprule
 & Base Covariates & Base + Rating Score & Base + 11 Qualities & Base + Both \\
\midrule
Science & 0.402611 & 0.514239 & 0.574734 & 0.617976 \\
Neurophysiology & 0.135365 & 0.330803 & 0.303937 & 0.345577 \\
Genetics & 0.190359 & 0.206164 & 0.284343 & 0.291995 \\
FASEB & 0.192006 & 0.260009 & 0.208367 & 0.353867 \\
Applied Physiology & 0.193243 & 0.433162 & 0.384400 & 0.459473 \\
All Journals & 0.403442 & 0.511649 & 0.575158 & 0.618178 \\
\bottomrule
\end{tabular}



# LLM TRAINING DATA ISSUE

In [6]:
# check results for different combinations of covariates
# try removing covariates that the LLM would be more likely to know 

def rf_chosen_vars(journal, variables, outcome = "Times cited (36mo)"):
    filename = f"open_access_results/{journal}_qualities.csv"
    results = pd.read_csv(filename)
    results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

        transform_df = pd.concat([
            results.loc[results[col] == 'Paper 1', 'Article Title.1'],
            results.loc[results[col] == 'Paper 2', 'Article Title.2']
        ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', f'{col}_score']

        journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
    filename = f"open_access_results/{journal}_basic.csv"
    results = pd.read_csv(filename)
    transform_df = pd.concat([
        results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
        results.loc[results['response'] == 'Paper 2', 'Article Title.2']
        ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', 'risk_score']
    rate_df.sort_values('Article Title').head()
    rate_df = rate_df[['Article Title', 'risk_score']]

    journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

    journal_df['log_authors'] = np.log(journal_df['Authors'])
    journal_df['log_pages'] = np.log(journal_df['Page length'])
    journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
    journal_df['log_outcome'] = np.log(journal_df[outcome] + 1)

    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived'] + variables]
    y = journal_df['log_outcome']

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   
        random_state=42
    )

    rf.fit(X, y)

    journal_df['oob_preds'] = rf.oob_prediction_
    oob_corr = journal_df['log_outcome'].corr(journal_df['oob_preds'])
    return oob_corr
    

'topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' 'applicability'

In [11]:
rf_results['Base + 11 Qualities']

Science               0.574734
Neurophysiology       0.303937
Genetics              0.284343
FASEB                 0.208367
Applied Physiology    0.384400
Name: Base + 11 Qualities, dtype: float64

In [29]:
# most stringent version is missing topic popularity, subfield popularity, and topic novelty

rows = []
for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
    journal_row = []
    for variables in [['writing_quality_score', 'technicality_score'], ['writing_quality_score', 'technicality_score', 'risk_score'], ['writing_quality_score', 'technicality_score', 'generalizability_score', 'applicability_score'], ['writing_quality_score', 'technicality_score', 'generalizability_score', 'applicability_score', 'title_catchiness_score'], ['writing_quality_score', 'technicality_score', 'generalizability_score', 'applicability_score', 'title_catchiness_score', 'impact_of_results_score', 'meaningful_contributions_score']]:
        oob_corr = rf_chosen_vars(journal, variables)
        journal_row.append(oob_corr) 
    rows.append(journal_row)

var_results = pd.DataFrame(rows, index = ['Science', 'Neurophysiology', 'Genetics', 'FASEB','Applied Physiology'], columns =['Technicality + Writing Score', 'Technicality, Writing, Rating Score', 'Technicality, Writing Quality, Generalizability, Applicability', 'Technicality, Writing Quality, Generalizability, Applicability, Title Catchiness','Technicality, Writing Quality, Generalizability, Applicability, Title Catchiness, Impact of Results, Meaningful Contributions'])
pd.concat([var_results, rf_results['Base + 11 Qualities']], axis = 1)


,Technicality + Writing Score,"Technicality, Writing, Rating Score","Technicality, Writing Quality, Generalizability, Applicability","Technicality, Writing Quality, Generalizability, Applicability, Title Catchiness","Technicality, Writing Quality, Generalizability, Applicability, Title Catchiness, Impact of Results, Meaningful Contributions",Base + 11 Qualities
Science,0.454664,0.585497,0.534842,0.533107,0.571680,0.574734
Neurophysiology,0.225933,0.346585,0.226453,0.313473,0.316031,0.303937
Genetics,0.172598,0.192274,0.292442,0.296149,0.281071,0.284343
FASEB,0.017298,0.297672,0.128951,0.145171,0.121543,0.208367
Applied Physiology,0.399476,0.453745,0.389937,0.390834,0.362938,0.384400


In [30]:
pd.concat([rf_results['Base Covariates'], rf_results['Base + Rating Score'], var_results, rf_results['Base + 11 Qualities']], axis = 1)

,Base Covariates,Base + Rating Score,Technicality + Writing Score,"Technicality, Writing, Rating Score","Technicality, Writing Quality, Generalizability, Applicability","Technicality, Writing Quality, Generalizability, Applicability, Title Catchiness","Technicality, Writing Quality, Generalizability, Applicability, Title Catchiness, Impact of Results, Meaningful Contributions",Base + 11 Qualities
Science,0.402611,0.514239,0.454664,0.585497,0.534842,0.533107,0.571680,0.574734
Neurophysiology,0.135365,0.330803,0.225933,0.346585,0.226453,0.313473,0.316031,0.303937
Genetics,0.190359,0.206164,0.172598,0.192274,0.292442,0.296149,0.281071,0.284343
FASEB,0.192006,0.260009,0.017298,0.297672,0.128951,0.145171,0.121543,0.208367
Applied Physiology,0.193243,0.433162,0.399476,0.453745,0.389937,0.390834,0.362938,0.384400


In [19]:
# get variable importance for each random forest on all 10 covariates 

def get_var_importance(journal, outcome = "Times cited (36mo)"): 
    filename = f"open_access_results/{journal}_qualities.csv"
    results = pd.read_csv(filename)
    results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

        transform_df = pd.concat([
            results.loc[results[col] == 'Paper 1', 'Article Title.1'],
            results.loc[results[col] == 'Paper 2', 'Article Title.2']
        ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', f'{col}_score']

        journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
    filename = f"open_access_results/{journal}_basic.csv"
    results = pd.read_csv(filename)
    transform_df = pd.concat([
        results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
        results.loc[results['response'] == 'Paper 2', 'Article Title.2']
        ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', 'risk_score']
    rate_df.sort_values('Article Title').head()
    rate_df = rate_df[['Article Title', 'risk_score']]

    journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

    journal_df['log_authors'] = np.log(journal_df['Authors'])
    journal_df['log_pages'] = np.log(journal_df['Page length'])
    journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
    journal_df['log_outcome'] = np.log(journal_df[outcome] + 1)

    X = journal_df[['log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score']]
    y = journal_df['log_outcome']

    rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   
        random_state=42
    )

    rf.fit(X, y)
    importances = rf.feature_importances_

    importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)
    return importance_df

In [21]:
test = get_var_importance("science")

In [27]:
full = pd.DataFrame({'order': range(1,15 )})
for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
    importance_df = get_var_importance(journal)
    importance_df.columns = [f'{journal}_feature', f'{journal}_importance']
    importance_df['order'] = range(1, 15)
    full = full.merge(importance_df, on = 'order', how = 'left')

In [28]:
full

,order,science_feature,science_importance,neuro_feature,neuro_importance,genetics_feature,genetics_importance,faseb_feature,faseb_importance,physio_feature,physio_importance
0,1,writing_quality_score,0.198109,title_catchiness_score,0.157435,generalizability_score,0.149313,generalizability_score,0.193866,meaningful_contributions_score,0.242849
1,2,log_authors,0.138165,writing_quality_score,0.127879,impact_of_results_score,0.144191,impact_of_results_score,0.117498,generalizability_score,0.091885
2,3,impact_of_results_score,0.122011,impact_of_results_score,0.116941,meaningful_contributions_score,0.128975,title_catchiness_score,0.088946,writing_quality_score,0.089942
3,4,log_pages,0.109399,meaningful_contributions_score,0.109730,technicality_score,0.111145,log_pages,0.088863,impact_of_results_score,0.087002
4,5,applicability_score,0.100272,applicability_score,0.084900,log_pages,0.075131,writing_quality_score,0.077323,title_catchiness_score,0.078153
5,6,generalizability_score,0.066847,generalizability_score,0.083979,writing_quality_score,0.073868,technicality_score,0.075051,subfield_popularity_score,0.063326
6,7,meaningful_contributions_score,0.059957,subfield_popularity_score,0.059314,title_catchiness_score,0.068563,applicability_score,0.069430,technicality_score,0.061434
7,8,technicality_score,0.051371,technicality_score,0.058505,subfield_popularity_score,0.063734,subfield_popularity_score,0.062389,log_pages,0.061298
8,9,title_catchiness_score,0.044496,log_pages,0.058178,applicability_score,0.061680,topic_novelty_score,0.061866,applicability_score,0.060643
9,10,subfield_popularity_score,0.037464,log_authors,0.056642,log_authors,0.057331,log_authors,0.057136,log_authors,0.045983


In [40]:
# combine results into one dataframe and run all together 

all_journals = pd.DataFrame()

for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
    filename = f"open_access_results/{journal}_qualities.csv"
    results = pd.read_csv(filename)
    results[['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']] = results['response'].str.split(',', expand=True, n =10).apply(lambda lst: [x.strip() for x in lst])
    journal_df = pd.read_csv(f"open_access_journal_data/{journal}.csv")
    for col in ['topic_novelty', 'topic_popularity', 'title_catchiness', 'generalizability', 'writing_quality', 'impact_of_results', 'subfield_popularity', 'technicality', 'meaningful_contributions' ,'journal_fit', 'applicability']:

        transform_df = pd.concat([
            results.loc[results[col] == 'Paper 1', 'Article Title.1'],
            results.loc[results[col] == 'Paper 2', 'Article Title.2']
        ])

        rate_df = transform_df.value_counts().reset_index()
        rate_df.columns = ['Article Title', f'{col}_score']

        journal_df = pd.merge(journal_df, rate_df, on = 'Article Title', how = 'left').fillna(0)
    filename = f"open_access_results/{journal}_basic.csv"
    results = pd.read_csv(filename)
    transform_df = pd.concat([
        results.loc[results['response'] == 'Paper 1', 'Article Title.1'],
        results.loc[results['response'] == 'Paper 2', 'Article Title.2']
        ])

    rate_df = transform_df.value_counts().reset_index()
    rate_df.columns = ['Article Title', 'risk_score']
    rate_df.sort_values('Article Title').head()
    rate_df = rate_df[['Article Title', 'risk_score']]

    journal_df = pd.merge(journal_df, rate_df, on = "Article Title", how = 'left').fillna(0)

    journal_df['log_authors'] = np.log(journal_df['Authors'])
    journal_df['log_pages'] = np.log(journal_df['Page length'])
    journal_df['review'] = np.where(journal_df['Article type'] == "Review", 1, 0) 
    journal_df['log_outcome'] = np.log(journal_df["Times cited (36mo)"] + 1)
    journal_df['journal'] = journal

    all_journals = pd.concat([all_journals, journal_df])

    all_journals['science'] = np.where(all_journals['journal'] == "science", 1, 0)
    all_journals['neuro'] = np.where(all_journals['journal'] == "neuro", 1, 0)
    all_journals['genetics'] = np.where(all_journals['journal'] == "genetics", 1, 0)
    all_journals['faseb'] = np.where(all_journals['journal'] == "faseb", 1, 0)

In [43]:
all_journals.tail()

,Journal,Year,Volume,Issue,Beginning page,Article Title,Authors,Article type,No. References,Page length,...,log_pages,review,log_outcome,journal,science,neuro,genetics,faseb,Unnamed: 0.1,Unnamed: 0
196,J. Appl. Physiol.,2007,102,4,1671,Modulation of glucose transport in skeletal mu...,1,Review,79,6,...,1.791759,1,2.890372,physio,0,0,0,0,196.0,2871.0
197,J. Appl. Physiol.,2007,102,4,1677,The role of free radicals in the pathophysiolo...,2,Review,95,10,...,2.302585,1,3.295837,physio,0,0,0,0,197.0,2872.0
198,J. Appl. Physiol.,2007,102,4,1687,Ventilatory muscle activation and inflammation...,2,Review,86,9,...,2.197225,1,1.945910,physio,0,0,0,0,198.0,2873.0
199,J. Appl. Physiol.,2007,102,4,1696,8-Oxoguanosine and uracil repair of nuclear an...,4,Article,36,6,...,1.791759,0,2.197225,physio,0,0,0,0,199.0,2874.0
200,J. Appl. Physiol.,2007,102,4,1702,Intermittent hyperthermia enhances skeletal mu...,6,Article,48,6,...,1.791759,0,2.397895,physio,0,0,0,0,200.0,2875.0


In [45]:
X = all_journals[['science', 'neuro', 'genetics', 'faseb', 'log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score']]
y = all_journals['log_outcome']

rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   
        random_state=42
    )

rf.fit(X, y)
importances = rf.feature_importances_

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(importance_df)

                           feature  importance
0                          science    0.503159
12           writing_quality_score    0.061078
13         impact_of_results_score    0.050060
16  meaningful_contributions_score    0.045587
11          generalizability_score    0.045287
4                      log_authors    0.042272
5                        log_pages    0.039306
15              technicality_score    0.038278
17             applicability_score    0.037444
10          title_catchiness_score    0.031579
8              topic_novelty_score    0.026952
3                            faseb    0.026920
14       subfield_popularity_score    0.023660
9           topic_popularity_score    0.020792
6                           review    0.003655
2                         genetics    0.001974
7                    Self-archived    0.001090
1                            neuro    0.000909


In [48]:
X2 = all_journals[['log_authors', 'log_pages', 'review', 'Self-archived', 'topic_novelty_score',
       'topic_popularity_score', 'title_catchiness_score',
       'generalizability_score', 'writing_quality_score',
       'impact_of_results_score', 'subfield_popularity_score',
       'technicality_score', 'meaningful_contributions_score', 'applicability_score']]
y = all_journals['log_outcome']

rf = RandomForestRegressor(
        n_estimators=500,
        oob_score=True,
        bootstrap=True,   
        random_state=42
    )

rf.fit(X2, y)
importances = rf.feature_importances_

importance_df = pd.DataFrame({
    "feature": X2.columns,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(importance_df)

                           feature  importance
1                        log_pages    0.338843
8            writing_quality_score    0.117955
13             applicability_score    0.071101
9          impact_of_results_score    0.065692
7           generalizability_score    0.062580
11              technicality_score    0.062263
0                      log_authors    0.059121
12  meaningful_contributions_score    0.056365
10       subfield_popularity_score    0.042756
4              topic_novelty_score    0.041858
5           topic_popularity_score    0.040034
6           title_catchiness_score    0.036671
2                           review    0.003802
3                    Self-archived    0.000959


Some things we notice:

1. Topic popularity is consistently towward the bottom (yay!)
2. Subfield popularity is around the middle
3. Science actually does very well with just technicality and writing score, which is a good sign! It outperforms the citation pair score once it has technicality, writing score, generalizability, and applicability.
4. Only Science and Genetics have the covariates outperform the citation score alone, the other three do not. 

In [26]:
# GET STANDARD ERRORS
import statsmodels.api as sm
def get_ses(journal): 
    if journal in ['science', 'neuro', 'genetics', 'faseb', 'physio']:
        journal_df = run_random_forests(journal)
    elif journal == "all":
        journal_df = run_rf_full()
    y = journal_df['log_outcome']
    X1 = sm.add_constant(journal_df[['Open Access', 'oob_preds_neither']])
    X2 = sm.add_constant(journal_df[['Open Access', 'oob_preds_basic']])
    X3 = sm.add_constant(journal_df[['Open Access', 'oob_preds_qualities']])
    X4 = sm.add_constant(journal_df[['Open Access', 'oob_preds_both']])

    model1 = sm.OLS(y, X1).fit()
    model2 = sm.OLS(y, X2).fit()
    model3 = sm.OLS(y, X3).fit()
    model4 = sm.OLS(y, X4).fit()

    return [model1.bse['Open Access'], model2.bse['Open Access'], model3.bse['Open Access'], model4.bse['Open Access']]


In [29]:
rows = []
for journal in ['science', 'neuro', 'genetics', 'faseb', 'physio', "all"]:
    row = get_ses(journal)
    rows.append(row)

se_results = pd.DataFrame(rows, index = ['Science', 'Neurophysiology', 'Genetics', 'FASEB','Applied Physiology', "All Journals"], columns =['Base Covariates', 'Base + Rating Score', 'Base + 11 Qualities', 'Base + Both'] )
print(se_results)

                    Base Covariates  Base + Rating Score  Base + 11 Qualities  \
Science                    0.121024             0.113333             0.108191   
Neurophysiology            0.111322             0.105958             0.106967   
Genetics                   0.102855             0.102338             0.100379   
FASEB                      0.095469             0.093696             0.095051   
Applied Physiology         0.160876             0.149599             0.152021   
All Journals               0.120976             0.113538             0.108160   

                    Base + Both  
Science                0.103858  
Neurophysiology        0.105366  
Genetics               0.100048  
FASEB                  0.091012  
Applied Physiology     0.146924  
All Journals           0.103842  


In [30]:
print(se_results.to_latex())

\begin{tabular}{lrrrr}
\toprule
 & Base Covariates & Base + Rating Score & Base + 11 Qualities & Base + Both \\
\midrule
Science & 0.121024 & 0.113333 & 0.108191 & 0.103858 \\
Neurophysiology & 0.111322 & 0.105958 & 0.106967 & 0.105366 \\
Genetics & 0.102855 & 0.102338 & 0.100379 & 0.100048 \\
FASEB & 0.095469 & 0.093696 & 0.095051 & 0.091012 \\
Applied Physiology & 0.160876 & 0.149599 & 0.152021 & 0.146924 \\
All Journals & 0.120976 & 0.113538 & 0.108160 & 0.103842 \\
\bottomrule
\end{tabular}



For each journal, we:

1. Fit four random forests (with base covariates, base + pair score, base + 10 qualities, base + both)
2. Take the OOB preds from this and run a linear regression against the outcome with those oob preds + treatment indicator. Present SEs of the treatment indicator and see how that gets smaller as we add more useful covariates. 

In [ ]:
# convert into essential sample size 